# Quick start: attributions and interactions across a Rashomon set

Train a reference model, sample its mask-based Rashomon set, and look at how feature main effects and feature interaction scores (FIS) vary across the set.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor

from generalized_rashomon_set import RashomonExplainer, plots

## A synthetic regression with one true interaction

`y = x0 + x1 + 2 x0 x1 + 0.5 x2 + noise`; `x3` is pure noise.

In [2]:
rng = np.random.default_rng(0)
X = pd.DataFrame(rng.normal(size=(1500, 4)), columns=['x0', 'x1', 'x2', 'x3'])
y = X.x0 + X.x1 + 2 * X.x0 * X.x1 + 0.5 * X.x2 + rng.normal(scale=0.1, size=len(X))
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)
model = MLPRegressor((64, 64), max_iter=2000, random_state=0).fit(X_train, y_train)

## Explain

`fit()` runs the four stages: reference attributions, greedy mask search, attributions of every sampled model, and the FIS of every joint-masked pair model. `n_jobs=-1` uses all cores.

In [3]:
explainer = RashomonExplainer(model, X_test, y_test, loss='mse', epsilon_rate=0.1, delta=0.1,
                              shuffle_times=30, n_jobs=-1, random_state=0, verbose=True)
explainer.fit()

2026-10-20 17:40:57,203 INFO reference loss mean_squared_error=0.0256369, epsilon=0.00256369, 4 features, 450 samples


2026-10-20 17:40:59,768 INFO reference explained: 4 features, 6 subsets (2.6s)


2026-10-20 17:41:00,676 INFO sampled 88 models (0.9s)


2026-10-20 17:41:01,714 INFO attributions of 88 distinct sampled models (1.0s)


2026-10-20 17:41:02,845 INFO interactions: 6 subsets x 36 models (1.1s)


In [4]:
explainer.attribution_summary().round(4)

,feature,name,reference,min,max,range,mean,std,n_models
0,0,x0,8.2749,7.9982,8.9267,0.9285,8.3879,0.2703,22
1,1,x1,8.3411,8.0260,8.8705,0.8445,8.4131,0.2481,22
2,2,x2,0.4864,0.4412,0.5479,0.1067,0.4920,0.0382,22
3,3,x3,-0.0004,-0.0011,0.0053,0.0064,0.0022,0.0024,22


In [5]:
explainer.interaction_summary().round(4)

,subset,name,reference,min,max,range,mean,std,n_models,frac_in_set
0,"(0, 1)",x0 x x1,-6.7922,-7.5626,-6.2583,1.3043,-6.8969,0.3360,36,0.5000
1,"(0, 2)",x0 x x2,-0.1380,-0.4178,0.7865,1.2043,0.0040,0.2430,36,0.5000
2,"(0, 3)",x0 x x3,0.0173,-0.4097,0.3602,0.7699,-0.0209,0.2004,36,0.5000
3,"(1, 2)",x1 x x2,0.1977,-0.4936,0.7632,1.2569,-0.0120,0.2693,36,0.5000
4,"(1, 3)",x1 x x3,-0.0617,-0.5563,0.5146,1.0708,-0.0547,0.2413,36,0.5000
5,"(2, 3)",x2 x x3,0.0046,-0.0164,0.0115,0.0279,-0.0025,0.0068,36,0.4444


## Plots

In [6]:
plots.swarm_plot_attribution(explainer, boxplot=True);

In [7]:
plots.swarm_plot_interaction(explainer, boxplot=True);

/Users/sichaoli/miniconda/miniconda3/lib/python3.12/site-packages/seaborn/categorical.py:3399: UserWarning: 58.3% of the points cannot be placed; you may want to decrease the size of the markers or use stripplot.
  warnings.warn(msg, UserWarning)


In [8]:
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
plots.halo_plot(explainer, ('x0', 'x1'), ax=axes[0])   # strong interaction: empirical halo far from the circle
plots.halo_plot(explainer, ('x0', 'x2'), ax=axes[1])   # no interaction: halos coincide
plt.tight_layout()

In [9]:
plots.search_trace_plot(explainer);

## Several Rashomon radii in one plot

Pass `radii` to `explain_interactions` to evaluate pair models whose single-feature loss budgets add up to a fraction of `epsilon`.

In [10]:
explainer.explain_interactions(order=2, radii=(0.2, 0.4, 0.6, 0.8, 1.0))
plots.halo_plot(explainer, ('x0', 'x1'));

2026-10-20 17:41:06,248 INFO interactions: 6 subsets x 100 models (3.0s)


## Save and reload

In [11]:
explainer.save('quickstart_results.json')
reloaded = RashomonExplainer.load('quickstart_results.json', model, X_test, y_test)
reloaded.interaction_summary().head()

2026-10-20 17:41:06,264 INFO saved results to quickstart_results.json


2026-10-20 17:41:06,267 INFO reference loss mean_squared_error=0.0256369, epsilon=0.00256369, 4 features, 450 samples


,subset,name,reference,min,max,range,mean,std,n_models,frac_in_set
0,"(0, 1)",x0 x x1,-6.792151,-7.587179,-6.091970,1.495209,-6.874822,0.300276,100,0.66
1,"(0, 2)",x0 x x2,-0.138032,-0.519162,0.781868,1.301030,-0.034682,0.213423,100,0.82
2,"(0, 3)",x0 x x3,0.017277,-0.420146,0.409861,0.830007,-0.012640,0.198239,100,0.82
3,"(1, 2)",x1 x x2,0.197745,-0.576495,0.757923,1.334418,-0.034984,0.247261,100,0.82
4,"(1, 3)",x1 x x3,-0.061733,-0.596144,0.635570,1.231714,-0.008858,0.241326,100,0.81
